In [1]:
!git clone https://github.com/AzxRevanth/ano-research

Cloning into 'ano-research'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 24 (delta 1), reused 0 (delta 0), pack-reused 16 (from 1)
Receiving objects: 100% (24/24), 103.32 MiB | 12.63 MiB/s, done.
Resolving deltas: 100% (3/3), done.
Updating files: 100% (12/12), done.


In [2]:
%cd ano-research

/content/ano-research


In [3]:
!pip install pyod

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.6/58.6 kB 832.6 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.2/412.2 kB 3.8 MB/s eta 0:00:00


In [4]:
import pandas as pd
import numpy as np
from sklearn.decomposition import NMF
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, f1_score, precision_score, recall_score

df = pd.read_csv('pipeline_cache/ampds_behavior_context_labeled_features.csv')

# column groups
label_cols = ['is_anomaly', 'anomaly_type']
meta_cols = ['window_id']
context_cols = ['Temp (C)', 'Rel Hum (%)', 'Wind Spd (km/h)', 'Visibility (km)',
                'Stn Press (kPa)', 'wx_Clear', 'wx_Cloudy', 'wx_Fog', 'wx_Rain',
                'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend']

# rest are behavior features
behavior_cols = [c for c in df.columns if c not in label_cols + meta_cols + context_cols]

print("Behavior features:", len(behavior_cols))
print("Context features:", len(context_cols))
print("Anomaly count:", df['is_anomaly'].sum())

X_behavior = df[behavior_cols].values
y_true = df['is_anomaly'].values

scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_behavior)

print("\nShape of behavior matrix:", X_scaled.shape)
print("Any negatives after scaling:", (X_scaled < 0).any())

Behavior features: 336
Context features: 14
Anomaly count: 92

Shape of behavior matrix: (5856, 336)
Any negatives after scaling: False


In [5]:
# Task A : Global NMF
results = []

for n_components in [3, 5, 8, 10]:
    model = NMF(n_components=n_components, init='nndsvda', random_state=42, max_iter=500)
    W = model.fit_transform(X_scaled)
    H = model.components_

    # reconstruction residual = anomaly score
    X_reconstructed = W @ H
    residuals = np.mean((X_scaled - X_reconstructed) ** 2, axis=1)

    auc = roc_auc_score(y_true, residuals)

    # threshold at 95th percentile of normal windows
    threshold = np.percentile(residuals[y_true == 0], 95)
    y_pred = (residuals > threshold).astype(int)

    f1 = f1_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)

    results.append({
        'n_components': n_components,
        'auc_roc': round(auc, 3),
        'f1': round(f1, 3),
        'precision': round(precision, 3),
        'recall': round(recall, 3)
    })
    print(f"K={n_components} — AUC: {auc:.3f}, F1: {f1:.3f}, Precision: {precision:.3f}, Recall: {recall:.3f}")

results_df = pd.DataFrame(results)
print("\nSummary:")
print(results_df)

K=3 — AUC: 0.666, F1: 0.066, Precision: 0.043, Recall: 0.141


/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


K=5 — AUC: 0.692, F1: 0.061, Precision: 0.040, Recall: 0.130
K=8 — AUC: 0.679, F1: 0.081, Precision: 0.052, Recall: 0.174
K=10 — AUC: 0.655, F1: 0.061, Precision: 0.040, Recall: 0.130

Summary:
   n_components  auc_roc     f1  precision  recall
0             3    0.666  0.066      0.043   0.141
1             5    0.692  0.061      0.040   0.130
2             8    0.679  0.081      0.052   0.174
3            10    0.655  0.061      0.040   0.130


/usr/local/lib/python3.12/dist-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(


In [7]:
from pyod.models.iforest import IForest
from pyod.models.copod import COPOD
from pyod.models.ecod import ECOD
from pyod.models.hbos import HBOS
from pyod.models.lof import LOF

contamination = 92 / 5856  # actual anomaly rate
print(f"Using contamination: {contamination:.4f}")

pyod_models = {
    'IForest': IForest(contamination=contamination, random_state=42),
    'COPOD': COPOD(contamination=contamination),
    'ECOD': ECOD(contamination=contamination),
    'HBOS': HBOS(contamination=contamination),
    'LOF': LOF(contamination=contamination),
}

# three feature sets to compare
feature_sets = {
    'Behavior only': X_scaled,
    'Behavior + Global NMF residual': np.column_stack([X_scaled, residuals_global]),
    'Behavior + Context NMF residual': np.column_stack([X_scaled, residuals_context]),
}

pyod_results = []

for feat_name, X_feat in feature_sets.items():
    for algo_name, model in pyod_models.items():
        m = model.__class__(**model.get_params())
        m.fit(X_feat)

        auc = roc_auc_score(y_true, m.decision_scores_)
        y_pred = m.labels_
        f1 = f1_score(y_true, y_pred, zero_division=0)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)

        pyod_results.append({
            'features': feat_name,
            'algorithm': algo_name,
            'auc_roc': round(auc, 3),
            'f1': round(f1, 3),
            'precision': round(precision, 3),
            'recall': round(recall, 3),
        })
        print(f"{feat_name} | {algo_name} — AUC: {auc:.3f}, F1: {f1:.3f}")

pyod_df = pd.DataFrame(pyod_results)
pyod_df.to_csv('nmf_pyod_results.csv', index=False)

# summary pivot
pivot = pyod_df.pivot_table(index='algorithm', columns='features', values='auc_roc')
print("\nAUC-ROC Summary:")
print(pivot.to_string())

Using contamination: 0.0157
Behavior only | IForest — AUC: 0.705, F1: 0.022
Behavior only | COPOD — AUC: 0.716, F1: 0.043
Behavior only | ECOD — AUC: 0.673, F1: 0.043
Behavior only | HBOS — AUC: 0.733, F1: 0.022
Behavior only | LOF — AUC: 0.647, F1: 0.098
Behavior + Global NMF residual | IForest — AUC: 0.711, F1: 0.022
Behavior + Global NMF residual | COPOD — AUC: 0.717, F1: 0.043
Behavior + Global NMF residual | ECOD — AUC: 0.673, F1: 0.043
Behavior + Global NMF residual | HBOS — AUC: 0.733, F1: 0.022
Behavior + Global NMF residual | LOF — AUC: 0.647, F1: 0.098
Behavior + Context NMF residual | IForest — AUC: 0.711, F1: 0.022
Behavior + Context NMF residual | COPOD — AUC: 0.717, F1: 0.043
Behavior + Context NMF residual | ECOD — AUC: 0.674, F1: 0.043
Behavior + Context NMF residual | HBOS — AUC: 0.733, F1: 0.022
Behavior + Context NMF residual | LOF — AUC: 0.647, F1: 0.098

AUC-ROC Summary:
features   Behavior + Context NMF residual  Behavior + Global NMF residual  Behavior only
algor

In [8]:
from pyod.models.iforest import IForest
from pyod.models.copod import COPOD
from pyod.models.ecod import ECOD
from pyod.models.hbos import HBOS
from pyod.models.lof import LOF

contamination = 92 / 5856  # actual anomaly rate
print(f"Using contamination: {contamination:.4f}")

pyod_models = {
    'IForest': IForest(contamination=contamination, random_state=42),
    'COPOD': COPOD(contamination=contamination),
    'ECOD': ECOD(contamination=contamination),
    'HBOS': HBOS(contamination=contamination),
    'LOF': LOF(contamination=contamination),
}

# three feature sets to compare
feature_sets = {
    'Behavior only': X_scaled,
    'Behavior + Global NMF residual': np.column_stack([X_scaled, residuals_global]),
    'Behavior + Context NMF residual': np.column_stack([X_scaled, residuals_context]),
}

pyod_results = []

for feat_name, X_feat in feature_sets.items():
    for algo_name, model in pyod_models.items():
        m = model.__class__(**model.get_params())
        m.fit(X_feat)

        auc = roc_auc_score(y_true, m.decision_scores_)
        y_pred = m.labels_
        f1 = f1_score(y_true, y_pred, zero_division=0)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)

        pyod_results.append({
            'features': feat_name,
            'algorithm': algo_name,
            'auc_roc': round(auc, 3),
            'f1': round(f1, 3),
            'precision': round(precision, 3),
            'recall': round(recall, 3),
        })
        print(f"{feat_name} | {algo_name} — AUC: {auc:.3f}, F1: {f1:.3f}")

pyod_df = pd.DataFrame(pyod_results)
pyod_df.to_csv('nmf_pyod_results.csv', index=False)

# summary pivot
pivot = pyod_df.pivot_table(index='algorithm', columns='features', values='auc_roc')
print("\nAUC-ROC Summary:")
print(pivot.to_string())

Using contamination: 0.0157
Behavior only | IForest — AUC: 0.705, F1: 0.022
Behavior only | COPOD — AUC: 0.716, F1: 0.043
Behavior only | ECOD — AUC: 0.673, F1: 0.043
Behavior only | HBOS — AUC: 0.733, F1: 0.022
Behavior only | LOF — AUC: 0.647, F1: 0.098
Behavior + Global NMF residual | IForest — AUC: 0.711, F1: 0.022
Behavior + Global NMF residual | COPOD — AUC: 0.717, F1: 0.043
Behavior + Global NMF residual | ECOD — AUC: 0.673, F1: 0.043
Behavior + Global NMF residual | HBOS — AUC: 0.733, F1: 0.022
Behavior + Global NMF residual | LOF — AUC: 0.647, F1: 0.098
Behavior + Context NMF residual | IForest — AUC: 0.711, F1: 0.022
Behavior + Context NMF residual | COPOD — AUC: 0.717, F1: 0.043
Behavior + Context NMF residual | ECOD — AUC: 0.674, F1: 0.043
Behavior + Context NMF residual | HBOS — AUC: 0.733, F1: 0.022
Behavior + Context NMF residual | LOF — AUC: 0.647, F1: 0.098

AUC-ROC Summary:
features   Behavior + Context NMF residual  Behavior + Global NMF residual  Behavior only
algor

In [6]:
# save best global NMF residuals for later comparison
best_global_model = NMF(n_components=5, init='nndsvda', random_state=42, max_iter=2000)
W_global = best_global_model.fit_transform(X_scaled)
X_recon_global = W_global @ best_global_model.components_
residuals_global = np.mean((X_scaled - X_recon_global) ** 2, axis=1)
global_auc = roc_auc_score(y_true, residuals_global)
print(f"Best global NMF (K=5) AUC: {global_auc:.3f}")

# Task B — per context NMF
# weather regime is one-hot encoded so derive a single regime label
wx_cols = ['wx_Clear', 'wx_Cloudy', 'wx_Fog', 'wx_Rain']
df['weather_regime'] = df[wx_cols].idxmax(axis=1).str.replace('wx_', '')

regimes = df['weather_regime'].unique()
print("\nRegimes found:", regimes)
print("Regime distribution:\n", df['weather_regime'].value_counts())

# train one NMF per regime
context_models = {}
for regime in regimes:
    mask = df['weather_regime'] == regime
    X_regime = X_scaled[mask]

    if len(X_regime) < 20:
        print(f"Skipping {regime} — too few samples ({len(X_regime)})")
        continue

    model = NMF(n_components=5, init='nndsvda', random_state=42, max_iter=2000)
    model.fit(X_regime)
    context_models[regime] = model
    print(f"Trained NMF for {regime} — {mask.sum()} windows")

# score each window using its context expert
residuals_context = np.zeros(len(df))
for regime, model in context_models.items():
    mask = (df['weather_regime'] == regime).values
    X_regime = X_scaled[mask]
    W = model.transform(X_regime)
    X_recon = W @ model.components_
    residuals_context[mask] = np.mean((X_regime - X_recon) ** 2, axis=1)

context_auc = roc_auc_score(y_true, residuals_context)
threshold = np.percentile(residuals_context[y_true == 0], 95)
y_pred_context = (residuals_context > threshold).astype(int)
f1_context = f1_score(y_true, y_pred_context, zero_division=0)
precision_context = precision_score(y_true, y_pred_context, zero_division=0)
recall_context = recall_score(y_true, y_pred_context, zero_division=0)

print(f"\nPer-context NMF — AUC: {context_auc:.3f}, F1: {f1_context:.3f}, Precision: {precision_context:.3f}, Recall: {recall_context:.3f}")
print(f"Global NMF     — AUC: {global_auc:.3f}")
print(f"Improvement: {context_auc - global_auc:+.3f}")

Best global NMF (K=5) AUC: 0.693

Regimes found: ['Clear' 'Cloudy' 'Rain' 'Fog']
Regime distribution:
 weather_regime
Clear     2948
Cloudy    1876
Rain       652
Fog        380
Name: count, dtype: int64
Trained NMF for Clear — 2948 windows
Trained NMF for Cloudy — 1876 windows
Trained NMF for Rain — 652 windows
Trained NMF for Fog — 380 windows

Per-context NMF — AUC: 0.699, F1: 0.085, Precision: 0.056, Recall: 0.185
Global NMF     — AUC: 0.693
Improvement: +0.006


In [20]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
import numpy as np

for residuals, name in [(residuals_global, 'Global NMF'),
                         (residuals_context, 'Per-context NMF')]:

    threshold = np.percentile(residuals[y_true == 0], 95)
    y_pred = (residuals > threshold).astype(int)

    auc = roc_auc_score(y_true, residuals)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)

    print(f"\n{name}")
    print(f"Anomalies detected: {y_pred.sum()} out of {len(y_pred)} windows")
    print(f"Actual anomalies:   {y_true.sum()}")
    print(f"AUC: {auc:.3f} | F1: {f1:.3f} | Precision: {precision:.3f} | Recall: {recall:.3f}")


Global NMF
Anomalies detected: 301 out of 5856 windows
Actual anomalies:   92
AUC: 0.693 | F1: 0.061 | Precision: 0.040 | Recall: 0.130

Per-context NMF
Anomalies detected: 306 out of 5856 windows
Actual anomalies:   92
AUC: 0.699 | F1: 0.085 | Precision: 0.056 | Recall: 0.185


In [9]:
from pyod.models.auto_encoder import AutoEncoder
from pyod.models.vae import VAE
from pyod.models.deep_svdd import DeepSVDD

contamination = 92 / 5856
print(f"Using contamination: {contamination:.4f}")

pyod_models = {
    'AutoEncoder': AutoEncoder(
        hidden_neuron_list=[128, 64, 32, 64, 128],
        epoch_num=30,
        batch_size=64,
        contamination=contamination,
        random_state=42,
        verbose=0
    ),
    'VAE': VAE(
        encoder_neuron_list=[128, 64, 32],
        decoder_neuron_list=[32, 64, 128],
        epoch_num=30,
        batch_size=64,
        contamination=contamination,
        random_state=42,
        verbose=0
    ),
}

feature_sets = {
    'Behavior only': X_scaled,
    'Behavior + Global NMF residual': np.column_stack([X_scaled, residuals_global]),
    'Behavior + Context NMF residual': np.column_stack([X_scaled, residuals_context]),
}

pyod_results = []

for feat_name, X_feat in feature_sets.items():
    for algo_name, model in pyod_models.items():
        print(f"Running {algo_name} on {feat_name}...")
        m = model.__class__(**model.get_params())
        m.fit(X_feat)

        auc = roc_auc_score(y_true, m.decision_scores_)
        y_pred = m.labels_
        f1 = f1_score(y_true, y_pred, zero_division=0)
        precision = precision_score(y_true, y_pred, zero_division=0)
        recall = recall_score(y_true, y_pred, zero_division=0)

        pyod_results.append({
            'features': feat_name,
            'algorithm': algo_name,
            'auc_roc': round(auc, 3),
            'f1': round(f1, 3),
            'precision': round(precision, 3),
            'recall': round(recall, 3),
        })
        print(f"Done — AUC: {auc:.3f}, F1: {f1:.3f}")

pyod_df = pd.DataFrame(pyod_results)
pyod_df.to_csv('nmf_pyod_dl_results.csv', index=False)

pivot = pyod_df.pivot_table(index='algorithm', columns='features', values='auc_roc')
print("\nAUC-ROC Summary:")
print(pivot.to_string())

Using contamination: 0.0157
Running AutoEncoder on Behavior only...
Done — AUC: 0.723, F1: 0.011
Running VAE on Behavior only...
Done — AUC: 0.689, F1: 0.054
Running AutoEncoder on Behavior + Global NMF residual...
Done — AUC: 0.715, F1: 0.011
Running VAE on Behavior + Global NMF residual...
Done — AUC: 0.698, F1: 0.043
Running AutoEncoder on Behavior + Context NMF residual...
Done — AUC: 0.718, F1: 0.011
Running VAE on Behavior + Context NMF residual...
Done — AUC: 0.698, F1: 0.033

AUC-ROC Summary:
features     Behavior + Context NMF residual  Behavior + Global NMF residual  Behavior only
algorithm                                                                                  
AutoEncoder                            0.718                           0.715          0.723
VAE                                    0.698                           0.698          0.689


In [10]:
# W matrix — compressed representation (5856 rows × 5 components)
W_df = pd.DataFrame(W_global, columns=[f'nmf_component_{i+1}' for i in range(W_global.shape[1])])
W_df.insert(0, 'window_id', df['window_id'].values)
W_df

,window_id,nmf_component_1,nmf_component_2,nmf_component_3,nmf_component_4,nmf_component_5
0,2012-09-01 07:00:00+00:00,0.182064,0.002315,0.001012,0.337004,0.051324
1,2012-09-01 07:15:00+00:00,0.190025,0.003338,0.000575,0.269608,0.043466
2,2012-09-01 07:30:00+00:00,0.189633,0.000862,0.000000,0.230081,0.130866
3,2012-09-01 07:45:00+00:00,0.188522,0.002299,0.000000,0.262834,0.077095
4,2012-09-01 08:00:00+00:00,0.183818,0.000000,0.000000,0.291059,0.130562
...,...,...,...,...,...,...
5851,2012-11-01 05:45:00+00:00,0.191141,0.005489,0.000000,0.224455,0.031403
5852,2012-11-01 06:00:00+00:00,0.185258,0.001138,0.000087,0.289656,0.105632
5853,2012-11-01 06:15:00+00:00,0.181031,0.000000,0.000000,0.280762,0.166461
5854,2012-11-01 06:30:00+00:00,0.172448,0.000000,0.000000,0.338220,0.209673
